# Advanced `frozenset` Tutorial — Guided Problems with Solutions

This notebook is a **second, independent set of advanced problems** on Python's `frozenset`.

The format is intentionally tutorial-oriented:

1. introduce a realistic problem;
2. identify the data semantics;
3. break the problem into small steps;
4. implement one step at a time;
5. verify assumptions with assertions;
6. discuss design trade-offs and common mistakes.

The examples use only the Python standard library.

## What makes `frozenset` interesting?

A normal `set` is mutable:

```python
s = {1, 2}
s.add(3)
```

A `frozenset` is immutable:

```python
fs = frozenset({1, 2})
```

That one difference has an important consequence: a `frozenset` can usually be **hashed**, so it can participate in structures that require hashable values.

Typical advanced uses include:

- dictionary keys;
- members of other sets;
- memoization keys;
- immutable graph states;
- canonical representations of unordered data;
- mathematical set families;
- database dependency calculations;
- search algorithms.

## The central design question

Before using `frozenset`, ask:

> Does this value represent an unordered collection where duplicate multiplicity is irrelevant?

If yes, `frozenset` may be a very good fit.

If **order matters**, a `tuple` is usually more appropriate.

If **duplicate counts matter**, `frozenset(items)` is usually wrong because duplicate values collapse.

In [1]:
a = frozenset([1, 2, 3])
b = frozenset([3, 2, 1])
c = frozenset([1, 1, 2, 3])

assert a == b
assert a == c

print(a)
print("hashable:", isinstance(hash(a), int))

frozenset({1, 2, 3})
hashable: True


---

# Problem 1 — Immutable snapshots and aliasing

Imagine a service that stores a user's enabled capabilities.

A dangerous implementation stores the caller's mutable set directly:

```python
profile["capabilities"] = capabilities
```

If the caller later mutates that set, the stored profile silently changes too.

We want a snapshot whose meaning cannot change after creation.

## Step 1 — Observe the aliasing problem

Two names can refer to the same mutable object.

We will intentionally demonstrate that behavior first.

In [2]:
incoming = {"read", "write"}

unsafe_profile = {
    "name": "alice",
    "capabilities": incoming,
}

incoming.add("delete")

print("Incoming:", incoming)
print("Stored:  ", unsafe_profile["capabilities"])

assert "delete" in unsafe_profile["capabilities"]

Incoming: {'write', 'delete', 'read'}
Stored:   {'write', 'delete', 'read'}


The profile changed because `unsafe_profile["capabilities"]` and `incoming` refer to the same mutable set.

For configuration snapshots, that can create very difficult bugs.

## Step 2 — Freeze the boundary

A useful best practice is:

> Accept flexible input, normalize it, then store an immutable internal representation.

Here, the immutable representation is a `frozenset`.

In [3]:
incoming = {"read", "write"}

safe_profile = {
    "name": "alice",
    "capabilities": frozenset(incoming),
}

incoming.add("delete")

print("Incoming:", incoming)
print("Stored:  ", safe_profile["capabilities"])

assert "delete" not in safe_profile["capabilities"]

Incoming: {'write', 'delete', 'read'}
Stored:   frozenset({'write', 'read'})


## Step 3 — Verify that mutation is blocked

The frozen snapshot is not merely a copy. It also communicates and enforces an immutability contract.

In [4]:
try:
    safe_profile["capabilities"].add("admin")
except AttributeError as exc:
    print("Expected failure:", exc)

Expected failure: 'frozenset' object has no attribute 'add'


### Takeaway

`frozenset` is useful at **API boundaries** when a collection should become a stable value.

This is often more important than hashability itself.

---

# Problem 2 — A set of teams

Suppose a project is represented by several teams, and each team is represented only by its members.

We want to store all unique teams in another set.

A normal set cannot contain another normal set because inner sets are unhashable.

## Step 1 — Why a set of sets fails

In [5]:
team = {"Ana", "Bo"}

try:
    bad = {team}
except TypeError as exc:
    print("Expected failure:", exc)

Expected failure: unhashable type: 'set'


## Step 2 — Freeze each inner team

Each team is itself an unordered collection, so `frozenset` preserves the correct semantics.

In [6]:
teams = {
    frozenset({"Ana", "Bo"}),
    frozenset({"Cara", "Dan"}),
    frozenset({"Bo", "Ana"}),  # duplicate logical team
}

print(teams)
assert len(teams) == 2

{frozenset({'Ana', 'Bo'}), frozenset({'Cara', 'Dan'})}


## Step 3 — Freeze the outer collection too

If the entire family of teams should also be a hashable immutable value, freeze the outer collection.

In [7]:
project_structure = frozenset(teams)

print(project_structure)
print("Hash:", hash(project_structure))

registry = {
    project_structure: "project-alpha"
}

assert registry[project_structure] == "project-alpha"

frozenset({frozenset({'Ana', 'Bo'}), frozenset({'Cara', 'Dan'})})
Hash: -6204509552803656217


### Takeaway

Nested `frozenset` values naturally model a **set of sets**.

This is especially useful for mathematical structures, partitions, graph components, and canonical groupings.

---

# Problem 3 — Canonical unordered pairs, with a trap

An undirected connection between `"A"` and `"B"` is the same as one between `"B"` and `"A"`.

That suggests:

```python
frozenset({"A", "B"})
```

But there is an important edge case: what happens to a self-loop such as `("A", "A")`?

## Step 1 — Canonicalize a normal undirected edge

In [8]:
e1 = frozenset(("A", "B"))
e2 = frozenset(("B", "A"))

assert e1 == e2
print(e1)

frozenset({'B', 'A'})


## Step 2 — Investigate the self-loop problem

A `frozenset` removes duplicates, so `("A", "A")` becomes a one-element set.

In [9]:
loop = frozenset(("A", "A"))

print(loop)
print("size:", len(loop))

assert loop == frozenset({"A"})

frozenset({'A'})
size: 1


That may be acceptable in some graph models, but it means the representation alone no longer records that two endpoint positions were supplied.

If self-loops are forbidden, validate them before freezing.

## Step 3 — Build a safe constructor

In [10]:
def undirected_edge(u, v):
    if u == v:
        raise ValueError("Self-loops are not allowed.")
    return frozenset((u, v))

assert undirected_edge("A", "B") == undirected_edge("B", "A")

try:
    undirected_edge("A", "A")
except ValueError as exc:
    print("Expected validation error:", exc)

Expected validation error: Self-loops are not allowed.


### Takeaway

Canonicalization is powerful, but it can discard information.

Always check whether the information being discarded is truly irrelevant to the domain.

---

# Problem 4 — Equality surprises: `1`, `1.0`, and `True`

Python considers some values of different types equal:

```python
1 == 1.0 == True
```

Their hashes also agree.

What does that imply for a `frozenset`?

## Step 1 — Observe Python's equality rules

In [11]:
print(1 == 1.0)
print(1 == True)
print(hash(1), hash(1.0), hash(True))

assert 1 == 1.0 == True
assert hash(1) == hash(1.0) == hash(True)

True
True
1 1 1


## Step 2 — Put those values into a `frozenset`

Because sets use equality and hashing to identify unique elements, these values collapse into one logical member.

In [12]:
values = frozenset({1, 1.0, True})

print(values)
print("length:", len(values))

assert len(values) == 1

frozenset({1})
length: 1


## Step 3 — Preserve type information explicitly

If type identity matters, encode it into the key.

In [13]:
def typed_value(value):
    return (type(value), value)

typed_values = frozenset(
    typed_value(value)
    for value in [1, 1.0, True]
)

print(typed_values)
assert len(typed_values) == 3

frozenset({(<class 'int'>, 1), (<class 'bool'>, True), (<class 'float'>, 1.0)})


### Takeaway

`frozenset` follows Python's ordinary equality and hashing semantics.

Do not assume that values of different types will remain distinct.

---

# Problem 5 — Canonical feature combinations

A machine-learning pipeline is compiled for a combination of feature flags.

The flags are unordered:

```python
{"normalize", "clip", "log"}
```

We want repeated requests for the same logical combination to reuse a cached artifact.

## Step 1 — Define the canonical signature

Because flag order and duplicate inputs should not matter, `frozenset` is a direct representation of the domain.

In [14]:
def feature_signature(flags):
    return frozenset(flags)

assert feature_signature(["normalize", "clip"]) ==            feature_signature(["clip", "normalize"])

assert feature_signature(["clip", "clip", "normalize"]) ==            feature_signature(["normalize", "clip"])

## Step 2 — Use the signature as a dictionary key

In [15]:
pipeline_cache = {}

def compile_pipeline(flags):
    signature = feature_signature(flags)

    if signature not in pipeline_cache:
        pipeline_cache[signature] = {
            "id": len(pipeline_cache) + 1,
            "flags": signature,
        }

    return pipeline_cache[signature]

p1 = compile_pipeline(["normalize", "clip"])
p2 = compile_pipeline(["clip", "normalize"])
p3 = compile_pipeline(["normalize"])

assert p1 is p2
assert p1 is not p3
assert len(pipeline_cache) == 2

print(pipeline_cache)

{frozenset({'clip', 'normalize'}): {'id': 1, 'flags': frozenset({'clip', 'normalize'})}, frozenset({'normalize'}): {'id': 2, 'flags': frozenset({'normalize'})}}


### Takeaway

A good cache key should encode **semantic identity**, not incidental input formatting.

---

# Problem 6 — Deduplicating logical clauses

In propositional logic, a clause such as:

```text
A OR ¬B OR C
```

does not depend on the order of its literals.

We can represent a literal as a pair:

```python
("A", True)    # A
("B", False)   # NOT B
```

and a clause as a `frozenset` of literals.

## Step 1 — Build a clause constructor

In [16]:
def clause(*literals):
    return frozenset(literals)

c1 = clause(("A", True), ("B", False), ("C", True))
c2 = clause(("C", True), ("A", True), ("B", False))

assert c1 == c2
print(c1)

frozenset({('A', True), ('B', False), ('C', True)})


## Step 2 — Detect tautological clauses

A clause is tautological if it contains both a variable and its negation.

For example:

```text
A OR ¬A OR B
```

is always true.

In [17]:
def is_tautology(c):
    return any(
        (name, not polarity) in c
        for name, polarity in c
    )

tautological = clause(("A", True), ("A", False), ("B", True))
normal = clause(("A", True), ("B", False))

assert is_tautology(tautological)
assert not is_tautology(normal)

print("tautological:", tautological)

tautological: frozenset({('A', True), ('B', True), ('A', False)})


## Step 3 — Store a formula as a set of unique clauses

In [18]:
formula = frozenset({
    c1,
    c2,  # duplicate logical clause
    normal,
})

print("Unique clauses:", len(formula))
assert len(formula) == 2

Unique clauses: 2


### Takeaway

`frozenset` is especially natural when Python data is modeling mathematical set semantics directly.

---

# Problem 7 — Compute the boundary of a vertex subset

Let an undirected graph be stored as frozen edges.

For a selected vertex subset `S`, its **edge boundary** consists of edges with exactly one endpoint inside `S`.

This is useful in graph cuts and network analysis.

## Step 1 — Prepare the graph

In [19]:
edges = frozenset({
    frozenset({"A", "B"}),
    frozenset({"A", "C"}),
    frozenset({"B", "D"}),
    frozenset({"C", "D"}),
    frozenset({"D", "E"}),
})

selected = frozenset({"A", "B"})

## Step 2 — Translate the definition into set logic

For a 2-vertex edge `e`, it crosses the boundary exactly when:

```python
len(e & selected) == 1
```

In [20]:
def edge_boundary(edges, selected):
    return frozenset(
        edge
        for edge in edges
        if len(edge & selected) == 1
    )

boundary = edge_boundary(edges, selected)

for edge in boundary:
    print(edge)

assert boundary == frozenset({
    frozenset({"A", "C"}),
    frozenset({"B", "D"}),
})

frozenset({'D', 'B'})
frozenset({'A', 'C'})


## Step 3 — Compute the neighboring outside vertices

Each boundary edge contains exactly one vertex outside `selected`.

In [21]:
outside_neighbors = frozenset().union(
    *(edge - selected for edge in boundary)
)

print(outside_neighbors)
assert outside_neighbors == frozenset({"C", "D"})

frozenset({'D', 'C'})


### Takeaway

Using immutable sets allows graph operations to be expressed with small, declarative set expressions.

---

# Problem 8 — Maximum clique with Bron–Kerbosch

A **clique** is a set of vertices where every pair is connected.

The Bron–Kerbosch algorithm is a classic recursive algorithm for enumerating maximal cliques.

`frozenset` is a good fit because each recursive state is conceptually an immutable set of vertices.

## Step 1 — Build an adjacency map

In [22]:
graph_edges = [
    ("A", "B"),
    ("A", "C"),
    ("B", "C"),
    ("B", "D"),
    ("C", "D"),
    ("B", "E"),
]

vertices = frozenset(
    vertex
    for edge in graph_edges
    for vertex in edge
)

adjacency = {
    vertex: frozenset(
        other
        for u, v in graph_edges
        for other in (
            [v] if u == vertex else
            [u] if v == vertex else
            []
        )
    )
    for vertex in vertices
}

adjacency

{'C': frozenset({'A', 'B', 'D'}),
 'E': frozenset({'B'}),
 'D': frozenset({'B', 'C'}),
 'A': frozenset({'B', 'C'}),
 'B': frozenset({'A', 'C', 'D', 'E'})}

## Step 2 — Understand the three recursive sets

Bron–Kerbosch maintains:

- `R`: vertices already in the current clique;
- `P`: candidates that could extend it;
- `X`: vertices already processed for this branch.

A maximal clique is found when both `P` and `X` are empty.

In [23]:
def bron_kerbosch(R, P, X, adjacency):
    if not P and not X:
        return frozenset({R})

    cliques = set()

    for v in tuple(P):
        cliques.update(
            bron_kerbosch(
                R | {v},
                P & adjacency[v],
                X & adjacency[v],
                adjacency,
            )
        )
        P = P - {v}
        X = X | {v}

    return frozenset(cliques)

## Step 3 — Enumerate maximal cliques

In [24]:
maximal_cliques = bron_kerbosch(
    frozenset(),
    vertices,
    frozenset(),
    adjacency,
)

for clique in sorted(
    maximal_cliques,
    key=lambda c: (-len(c), tuple(sorted(c)))
):
    print(clique)

frozenset({'B', 'A', 'C'})
frozenset({'D', 'B', 'C'})
frozenset({'E', 'B'})


## Step 4 — Extract a maximum clique

"Maximal" means it cannot be extended.

"Maximum" means it has the largest size among all cliques.

In [25]:
maximum_clique = max(maximal_cliques, key=len)

print("Maximum clique:", maximum_clique)
print("Size:", len(maximum_clique))

assert len(maximum_clique) == 3

Maximum clique: frozenset({'B', 'A', 'C'})
Size: 3


### Takeaway

Recursive algorithms often become easier to reason about when state values are immutable.

`frozenset` also makes accidental cross-branch mutation impossible.

---

# Problem 9 — Database functional-dependency closure

In relational database theory, a functional dependency can be written:

```text
AB -> C
```

meaning attributes `A` and `B` determine `C`.

Attribute collections are mathematical sets, so `frozenset` is a natural representation.

## Step 1 — Represent dependencies

Each dependency will be a pair:

```python
(left_attributes, right_attributes)
```

In [26]:
dependencies = (
    (frozenset({"A"}), frozenset({"B"})),
    (frozenset({"B"}), frozenset({"C"})),
    (frozenset({"A", "C"}), frozenset({"D"})),
    (frozenset({"D"}), frozenset({"E"})),
)

## Step 2 — Define attribute closure

Start with the attributes we already know.

Repeatedly apply any dependency whose left side is already known.

Stop when a full pass adds nothing new.

In [27]:
def attribute_closure(start, dependencies):
    closure = frozenset(start)

    while True:
        expanded = closure

        for left, right in dependencies:
            if left <= expanded:
                expanded = expanded | right

        if expanded == closure:
            return closure

        closure = expanded

## Step 3 — Compute `A+`

In [28]:
a_closure = attribute_closure({"A"}, dependencies)

print("A+ =", a_closure)
assert a_closure == frozenset({"A", "B", "C", "D", "E"})

A+ = frozenset({'D', 'A', 'C', 'E', 'B'})


## Step 4 — Test whether an attribute set is a superkey

A set is a superkey if its closure contains every attribute in the relation.

In [29]:
relation = frozenset({"A", "B", "C", "D", "E"})

def is_superkey(attributes, relation, dependencies):
    return relation <= attribute_closure(attributes, dependencies)

assert is_superkey({"A"}, relation, dependencies)
assert not is_superkey({"B"}, relation, dependencies)

print("A is a superkey:", is_superkey({"A"}, relation, dependencies))

A is a superkey: True


### Takeaway

`frozenset` is not merely a Python trick here—it matches the underlying mathematics exactly.

---

# Problem 10 — Find candidate keys

Continuing the database example, a **candidate key** is a minimal superkey:

- it determines all attributes;
- removing any attribute from it destroys that property.

For small relations, we can search all attribute subsets.

## Step 1 — Generate subsets in increasing size

Searching smaller subsets first helps us identify minimal keys early.

In [30]:
from itertools import combinations

def subsets_by_size(items):
    items = tuple(items)

    for size in range(len(items) + 1):
        for combo in combinations(items, size):
            yield frozenset(combo)

## Step 2 — Reject supersets of already-found keys

Once `{A}` is known to be a candidate key, any superset containing `A` cannot be minimal.

In [31]:
def candidate_keys(relation, dependencies):
    keys = []

    for candidate in subsets_by_size(relation):
        if any(key <= candidate for key in keys):
            continue

        if is_superkey(candidate, relation, dependencies):
            keys.append(candidate)

    return frozenset(keys)

## Step 3 — Run the search

In [32]:
keys = candidate_keys(relation, dependencies)

print(keys)
assert keys == frozenset({frozenset({"A"})})

frozenset({frozenset({'A'})})


### Complexity note

This brute-force method explores subsets and is exponential in the number of attributes.

It is suitable for teaching and small schemas, not large database-design automation.

---

# Problem 11 — Immutable search states for the classic river crossing

We will model a small state-space problem.

A farmer must move a wolf, goat, and cabbage across a river.

Unsafe situations:

- wolf with goat without farmer;
- goat with cabbage without farmer.

A state is fully described by which entities are on the left bank.

## Step 1 — Represent a state

The left bank is an unordered collection, so `frozenset` is appropriate.

Anything not on the left bank is on the right bank.

In [33]:
entities = frozenset({"farmer", "wolf", "goat", "cabbage"})

start_state = entities
goal_state = frozenset()

## Step 2 — Define safety

We check each bank separately.

In [34]:
def bank_is_safe(bank):
    if "farmer" in bank:
        return True

    if {"wolf", "goat"} <= bank:
        return False

    if {"goat", "cabbage"} <= bank:
        return False

    return True

def state_is_safe(left):
    right = entities - left
    return bank_is_safe(left) and bank_is_safe(right)

assert state_is_safe(start_state)

## Step 3 — Generate legal moves

The farmer always crosses.

The farmer may cross alone or with one entity located on the same bank.

In [35]:
def next_states(left):
    farmer_on_left = "farmer" in left
    current_bank = left if farmer_on_left else entities - left

    possible_passengers = [None] + [
        entity
        for entity in current_bank
        if entity != "farmer"
    ]

    results = set()

    for passenger in possible_passengers:
        moving = {"farmer"}

        if passenger is not None:
            moving.add(passenger)

        if farmer_on_left:
            candidate = left - moving
        else:
            candidate = left | moving

        if state_is_safe(candidate):
            results.add(frozenset(candidate))

    return frozenset(results)

## Step 4 — Breadth-first search

`frozenset` lets us put states directly into `visited`.

In [36]:
from collections import deque

def shortest_solution(start, goal):
    queue = deque([(start, [start])])
    visited = {start}

    while queue:
        state, path = queue.popleft()

        if state == goal:
            return path

        for nxt in next_states(state):
            if nxt not in visited:
                visited.add(nxt)
                queue.append((nxt, path + [nxt]))

    return None

solution_path = shortest_solution(start_state, goal_state)

assert solution_path is not None

for step, left in enumerate(solution_path):
    print(
        f"Step {step}:",
        "left =", set(left),
        "| right =", set(entities - left)
    )

Step 0: left = {'farmer', 'wolf', 'goat', 'cabbage'} | right = set()
Step 1: left = {'wolf', 'cabbage'} | right = {'farmer', 'goat'}
Step 2: left = {'farmer', 'wolf', 'cabbage'} | right = {'goat'}
Step 3: left = {'cabbage'} | right = {'farmer', 'wolf', 'goat'}
Step 4: left = {'farmer', 'goat', 'cabbage'} | right = {'wolf'}
Step 5: left = {'goat'} | right = {'farmer', 'wolf', 'cabbage'}
Step 6: left = {'farmer', 'goat'} | right = {'wolf', 'cabbage'}
Step 7: left = set() | right = {'farmer', 'wolf', 'goat', 'cabbage'}


### Takeaway

Search algorithms often need:

```python
visited = set()
```

Therefore, a hashable state representation can drastically simplify the implementation.

---

# Problem 12 — A robust recursive freezer

Memoization often fails when arguments contain dictionaries, lists, or sets because those containers are not hashable.

We want to convert nested data into an immutable hashable representation **without destroying its semantics**.

## Step 1 — Decide how each container should freeze

We will use these rules:

- dictionary: unordered mapping → `frozenset` of key/value pairs;
- set: unordered unique values → `frozenset`;
- list: ordered sequence → `tuple`;
- tuple: ordered sequence → `tuple`;
- scalar: keep as-is if hashable.

The important point is that we do **not** blindly turn everything into a `frozenset`.

In [37]:
from collections.abc import Mapping, Set as AbstractSet

def freeze(value):
    if isinstance(value, Mapping):
        return frozenset(
            (freeze(key), freeze(item))
            for key, item in value.items()
        )

    if isinstance(value, list):
        return tuple(freeze(item) for item in value)

    if isinstance(value, tuple):
        return tuple(freeze(item) for item in value)

    if isinstance(value, AbstractSet) and not isinstance(
        value, (str, bytes, bytearray)
    ):
        return frozenset(freeze(item) for item in value)

    hash(value)
    return value

## Step 2 — Verify dictionary order independence

In [38]:
left = {
    "filters": {"active": True, "regions": {"EU", "US"}},
    "columns": ["id", "name"],
}

right = {
    "columns": ["id", "name"],
    "filters": {"regions": {"US", "EU"}, "active": True},
}

assert freeze(left) == freeze(right)

## Step 3 — Verify list order preservation

In [39]:
a = {"columns": ["id", "name"]}
b = {"columns": ["name", "id"]}

assert freeze(a) != freeze(b)

## Step 4 — Use the frozen representation as a key

In [40]:
cache = {}

key = freeze(left)
cache[key] = "query-plan-17"

assert cache[freeze(right)] == "query-plan-17"

print(key)

frozenset({('columns', ('id', 'name')), ('filters', frozenset({('regions', frozenset({'EU', 'US'})), ('active', True)}))})


### Takeaway

Hashability should be added in a way that preserves the **meaning** of each nested container.

---

# Problem 13 — Memoization with semantic argument normalization

We will now build a memoizer using `freeze()`.

The goal is:

```python
f(config1)
f(config2)
```

to share a cache entry when the nested configurations are semantically equivalent.

## Step 1 — Construct a cache key

Positional argument order matters, so `args` stays tuple-like.

Keyword argument order does not matter, so the dictionary is frozen structurally.

In [41]:
from functools import wraps

def semantic_memoize(fn):
    cache = {}

    @wraps(fn)
    def wrapper(*args, **kwargs):
        key = (
            freeze(args),
            freeze(kwargs),
        )

        if key not in cache:
            cache[key] = fn(*args, **kwargs)

        return cache[key]

    wrapper.cache = cache
    return wrapper

## Step 2 — Decorate an expensive function

In [42]:
calls = 0

@semantic_memoize
def build_report(config):
    global calls
    calls += 1

    return {
        "rows": len(config["regions"]) * len(config["metrics"])
    }

## Step 3 — Call it with equivalent nested values

In [43]:
config1 = {
    "regions": {"EU", "US"},
    "metrics": ["revenue", "margin"],
}

config2 = {
    "metrics": ["revenue", "margin"],
    "regions": {"US", "EU"},
}

r1 = build_report(config1)
r2 = build_report(config2)

assert r1 == r2
assert calls == 1
assert len(build_report.cache) == 1

print(r1)
print("actual function calls:", calls)

{'rows': 4}
actual function calls: 1


### Important limitation

Our freezer defines semantic rules for common containers, but arbitrary custom objects may require their own canonicalization policy.

There is no universally correct way to "make any Python object hashable."

---

# Problem 14 — Antichains and incomparable sets

In a partially ordered set under subset inclusion, an **antichain** is a family where no member contains another.

Example:

```python
{1, 2}
{1, 3}
{2, 3}
```

These sets are pairwise incomparable by inclusion.

## Step 1 — Write an incomparability test

In [44]:
def incomparable(a, b):
    return not (a <= b or b <= a)

assert incomparable(
    frozenset({1, 2}),
    frozenset({1, 3}),
)

## Step 2 — Validate an entire family

In [45]:
def is_antichain(family):
    family = tuple(family)

    for i, left in enumerate(family):
        for right in family[i + 1:]:
            if not incomparable(left, right):
                return False

    return True

family1 = frozenset({
    frozenset({1, 2}),
    frozenset({1, 3}),
    frozenset({2, 3}),
})

family2 = frozenset({
    frozenset({1}),
    frozenset({1, 2}),
})

assert is_antichain(family1)
assert not is_antichain(family2)

## Step 3 — Build one greedy antichain

A greedy algorithm is not guaranteed to find the largest possible antichain, but it demonstrates how immutable set families can be manipulated.

In [46]:
candidates = [
    frozenset({1}),
    frozenset({2}),
    frozenset({1, 2}),
    frozenset({1, 3}),
    frozenset({2, 3}),
    frozenset({1, 2, 3}),
]

def greedy_antichain(candidates):
    chosen = []

    for candidate in sorted(candidates, key=len, reverse=True):
        if all(incomparable(candidate, existing) for existing in chosen):
            chosen.append(candidate)

    return frozenset(chosen)

greedy = greedy_antichain(candidates)

print(greedy)
assert is_antichain(greedy)

frozenset({frozenset({1, 2, 3})})


### Takeaway

A `frozenset` can represent both:

- one mathematical set;
- and, recursively, a mathematical family of sets.

---

# Problem 15 — Dependency signatures for incremental builds

Imagine a build system where an output depends on a set of source files.

We want a canonical signature such that source order does not matter.

Later, if dependencies change, we can compare signatures directly.

## Step 1 — Represent dependencies as a frozen value

In [47]:
def dependency_signature(paths):
    return frozenset(paths)

old = dependency_signature([
    "parser.py",
    "lexer.py",
    "tokens.py",
])

same_reordered = dependency_signature([
    "tokens.py",
    "parser.py",
    "lexer.py",
])

assert old == same_reordered

## Step 2 — Detect exactly what changed

In [48]:
new = dependency_signature([
    "parser.py",
    "lexer.py",
    "ast.py",
])

removed = old - new
added = new - old
changed = old ^ new

print("removed:", removed)
print("added:  ", added)
print("changed:", changed)

assert removed == frozenset({"tokens.py"})
assert added == frozenset({"ast.py"})

removed: frozenset({'tokens.py'})
added:   frozenset({'ast.py'})
changed: frozenset({'ast.py', 'tokens.py'})


## Step 3 — Use dependency signatures as cache keys

In [49]:
build_cache = {
    old: "artifact-v1",
    new: "artifact-v2",
}

assert build_cache[same_reordered] == "artifact-v1"

### Takeaway

`frozenset` is useful for **signatures** when membership matters but ordering does not.

---

# Problem 16 — Permission implication and minimal missing capabilities

A task requires a set of capabilities.

A user has another set.

We want to answer:

1. Is the user authorized?
2. Which capabilities are missing?
3. Which extra capabilities does the user have?

## Step 1 — Model requirements and possessions

In [50]:
required = frozenset({
    "read",
    "write",
    "approve",
})

possessed = frozenset({
    "read",
    "write",
    "audit",
})

## Step 2 — Authorization is a subset question

In [51]:
authorized = required <= possessed

print("authorized:", authorized)
assert not authorized

authorized: False


## Step 3 — Differences explain the decision

In [52]:
missing = required - possessed
extra = possessed - required

print("missing:", missing)
print("extra:  ", extra)

assert missing == frozenset({"approve"})
assert extra == frozenset({"audit"})

missing: frozenset({'approve'})
extra:   frozenset({'audit'})


## Step 4 — Rank users by how close they are

Suppose we have several users and want the one requiring the fewest additional capabilities.

In [53]:
users = {
    "alice": frozenset({"read"}),
    "bob": frozenset({"read", "write"}),
    "cara": frozenset({"read", "write", "audit"}),
}

gap_sizes = {
    user: len(required - capabilities)
    for user, capabilities in users.items()
}

best_user = min(gap_sizes, key=gap_sizes.get)

print(gap_sizes)
print("closest:", best_user)

assert best_user in {"bob", "cara"}
assert gap_sizes[best_user] == 1

{'alice': 2, 'bob': 1, 'cara': 1}
closest: bob


### Takeaway

Subset, difference, and symmetric-difference operations often provide clearer business logic than nested loops.

---

# Problem 17 — Immutable states in dynamic programming

Suppose we must choose projects under a dependency rule.

A state is the set of completed projects.

We want to count how many valid completion orders exist for a tiny dependency graph.

## Step 1 — Define dependencies

`dependencies[p]` is the set of projects that must already be complete before `p` can be selected.

In [54]:
project_dependencies = {
    "A": frozenset(),
    "B": frozenset({"A"}),
    "C": frozenset({"A"}),
    "D": frozenset({"B", "C"}),
}

all_projects = frozenset(project_dependencies)

## Step 2 — Determine currently available projects

A project is available when:

- it is not already complete;
- all its dependencies are complete.

In [55]:
def available_projects(completed):
    return frozenset(
        project
        for project, dependencies in project_dependencies.items()
        if project not in completed
        and dependencies <= completed
    )

assert available_projects(frozenset()) == frozenset({"A"})
assert available_projects(frozenset({"A"})) == frozenset({"B", "C"})

## Step 3 — Memoize by completed-project state

Because `completed` is a `frozenset`, it can be used directly as an `lru_cache` key.

In [56]:
from functools import lru_cache

@lru_cache(maxsize=None)
def count_orders(completed):
    if completed == all_projects:
        return 1

    return sum(
        count_orders(completed | {project})
        for project in available_projects(completed)
    )

total_orders = count_orders(frozenset())

print("Valid completion orders:", total_orders)
assert total_orders == 2

Valid completion orders: 2


## Step 4 — Inspect cache statistics

In [57]:
print(count_orders.cache_info())

CacheInfo(hits=1, misses=6, maxsize=None, currsize=6)


### Takeaway

Immutable state values combine especially well with memoized recursion.

---

# Problem 18 — Design a safe unordered-arguments memoizer

Sometimes a function is genuinely symmetric:

```python
max_distance(point_a, point_b)
```

Swapping the two points does not change the answer.

We want calls in either order to share one cache entry.

But we must preserve the identity of each point itself.

## Step 1 — Represent a point

A point is ordered `(x, y)`, so it should be a tuple—not a `frozenset`.

In [58]:
p1 = (0, 0)
p2 = (3, 4)

assert p1 != p2

## Step 2 — Represent the unordered pair of points

The pair of arguments is symmetric, so the outer container may be a `frozenset`.

In [59]:
def unordered_pair_key(a, b):
    return frozenset({a, b})

assert unordered_pair_key(p1, p2) == unordered_pair_key(p2, p1)

## Step 3 — Handle the duplicate-argument case carefully

`frozenset({a, a})` contains one item.

That is fine if `f(a, a)` has well-defined symmetric semantics, but it means the key does not preserve argument count by itself.

We can tag the arity explicitly.

In [60]:
def symmetric_binary_key(a, b):
    return (
        2,
        frozenset({a, b}),
        a == b,
    )

assert symmetric_binary_key(p1, p2) == symmetric_binary_key(p2, p1)
assert symmetric_binary_key(p1, p1) != symmetric_binary_key(p1, p2)

## Step 4 — Build the memoized function

In [61]:
distance_cache = {}

def squared_distance(a, b):
    key = symmetric_binary_key(a, b)

    if key not in distance_cache:
        ax, ay = a
        bx, by = b
        distance_cache[key] = (ax - bx) ** 2 + (ay - by) ** 2

    return distance_cache[key]

assert squared_distance(p1, p2) == 25
assert squared_distance(p2, p1) == 25
assert len(distance_cache) == 1

print(distance_cache)

{(2, frozenset({(3, 4), (0, 0)}), False): 25}


### Takeaway

Canonicalization should be **as aggressive as the function's semantics allow, but no more**.

---

# Summary — A disciplined way to choose `frozenset`

When deciding whether `frozenset` is appropriate, reason in this order.

## 1. Does order matter?

If yes, prefer an ordered immutable container such as `tuple`.

## 2. Do duplicate counts matter?

If yes, plain `frozenset(values)` is not enough.

You may need:

```python
tuple(sorted(values))
```

or a hashable frequency representation.

## 3. Does the value need stable identity-by-content?

If yes, `frozenset` is useful for:

- dictionary keys;
- cache keys;
- set membership;
- recursive search states;
- nested mathematical sets.

## 4. Are all members hashable?

A `frozenset` can only be hashable if its elements are hashable.

Nested mutable structures need semantic normalization first.

# Final challenge checklist

You should now be comfortable reasoning about:

- aliasing and immutable snapshots;
- nested set families;
- canonical unordered pairs;
- equality/hash collisions across compatible Python types;
- canonical configuration signatures;
- logic clauses;
- graph boundaries;
- maximal cliques;
- database attribute closures;
- candidate keys;
- state-space search;
- recursive structural freezing;
- semantic memoization;
- antichains;
- incremental dependency signatures;
- permission-set reasoning;
- dynamic programming with immutable states;
- symmetric-function cache keys.

# Comprehensive self-test

The following cell checks several important invariants from the notebook.

In [62]:
def self_test():
    # Unordered equality
    assert frozenset({1, 2, 3}) == frozenset({3, 2, 1})

    # Nested hashability
    nested = frozenset({
        frozenset({"A", "B"}),
        frozenset({"C"}),
    })
    hash(nested)

    # Edge canonicalization
    assert undirected_edge("X", "Y") == undirected_edge("Y", "X")

    # Typed-value distinction
    assert len(frozenset(
        typed_value(v)
        for v in [1, 1.0, True]
    )) == 3

    # Graph boundary
    assert edge_boundary(
        frozenset({
            frozenset({"A", "B"}),
            frozenset({"B", "C"}),
        }),
        frozenset({"A", "B"}),
    ) == frozenset({
        frozenset({"B", "C"})
    })

    # Attribute closure
    assert "E" in attribute_closure({"A"}, dependencies)

    # Recursive freezing semantics
    assert freeze({"x": [1, 2]}) != freeze({"x": [2, 1]})
    assert freeze({"a": 1, "b": 2}) == freeze({"b": 2, "a": 1})

    # DP answer
    assert count_orders(frozenset()) == 2

    # Symmetric cache
    assert squared_distance((0, 0), (3, 4)) ==                squared_distance((3, 4), (0, 0))

    print("All tutorial self-tests passed.")

self_test()

All tutorial self-tests passed.
